In [0]:
import json
import os
import time
from datetime import datetime, timezone
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Target locations (e.g., New York, London, Tokyo)
LOCATIONS = [
    {"city": "New York", "latitude": 40.7128, "longitude": -74.0060},
    {"city": "London", "latitude": 51.5074, "longitude": -0.1278},
    {"city": "Tokyo", "latitude": 35.6762, "longitude": 139.6503},
]

BASE_URL = "https://api.open-meteo.com/v1/forecast"
LANDING_ZONE = "landing_zone"


def create_session_with_retries() -> requests.Session:
    """Configures a requests session with exponential backoff for rate limits and server errors."""
    session = requests.Session()
    retry_strategy = Retry(
        total=5,
        backoff_factor=2,  # Waits 2s, 4s, 8s, 16s, 32s
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def fetch_weather_data(session: requests.Session, location: dict) -> dict:
    """Fetches hourly forecast data from Open-Meteo for a given coordinate pair."""
    params = {
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
        "timezone": "UTC",
        "forecast_days": 2,
    }
    response = session.get(BASE_URL, params=params, timeout=15)
    response.raise_for_status()
    payload = response.json()
    
    # Attach metadata for downstream tracking
    payload["metadata"] = {
        "city": location["city"],
        "ingested_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    return payload


def main():
    os.makedirs(LANDING_ZONE, exist_ok=True)
    session = create_session_with_retries()
    timestamp_str = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

    all_records = []
    for loc in LOCATIONS:
        try:
            print(f"Fetching weather data for {loc['city']}...")
            data = fetch_weather_data(session, loc)
            all_records.append(data)
            time.sleep(0.5)  # Polite pause between requests
        except requests.exceptions.RequestException as e:
            print(f"Failed to fetch data for {loc['city']}: {e}")

    output_path = os.path.join(LANDING_ZONE, f"weather_raw_{timestamp_str}.json")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(all_records, f, indent=2)

    print(f"Saved raw payload to {output_path}")


if __name__ == "__main__":
    main()

Fetching weather data for New York...
Fetching weather data for London...
Fetching weather data for Tokyo...
Saved raw payload to landing_zone/weather_raw_20261118_033422.json


Bronze

In [0]:
from pyspark.sql import functions as F

# Path to your uploaded JSON files
raw_path = "/Workspace/Users/cecp08@gmail.com/landing_zone/*.json"

# Read multiline JSON (since ingest.py saved a JSON array of city payloads)
df_raw = (
    spark.read.option("multiline", "true")
    .json(raw_path)
    .withColumn("bronze_loaded_at", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
)

# Write to Bronze Delta table (append-only raw history)
(
    df_raw.write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("workspace.default.bronze_weather_raw")
)

display(spark.table("workspace.default.bronze_weather_raw"))

elevation,generationtime_ms,hourly,hourly_units,latitude,longitude,metadata,timezone,timezone_abbreviation,utc_offset_seconds,bronze_loaded_at,source_file
32.0,0.0820159912109375,"List(List(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0), List(55, 64, 63, 58, 60, 64, 68, 69, 69, 75, 78, 78, 79, 82, 74, 65, 56, 48, 35, 28, 26, 30, 39, 48, 55, 56, 56, 61, 64, 66, 70, 70, 73, 72, 73, 76, 77, 79, 67, 62, 52, 40, 35, 38, 32, 40, 49, 56), List(10.1, 8.7, 7.9, 7.2, 6.6, 6.0, 5.6, 5.3, 5.0, 4.4, 4.2, 4.1, 3.8, 5.7, 8.4, 10.7, 12.7, 14.6, 16.6, 17.7, 17.5, 15.7, 12.5, 10.7, 9.5, 8.6, 7.7, 6.9, 6.3, 5.9, 5.7, 5.4, 5.0, 4.7, 4.3, 3.9, 3.5, 5.3, 8.6, 11.5, 13.7, 15.2, 16.5, 17.2, 17.5, 15.4, 12.4, 10.7), List(2026-11-18T00:00, 2026-11-18T01:00, 2026-11-18T02:00, 2026-11-18T03:00, 2026-11-18T04:00, 2026-11-18T05:00, 2026-11-18T06:00, 2026-11-18T07:00, 2026-11-18T08:00, 2026-11-18T09:00, 2026-11-18T10:00, 2026-11-18T11:00, 2026-11-18T12:00, 2026-11-18T13:00, 2026-11-18T14:00, 2026-11-18T15:00, 2026-11-18T16:00, 2026-11-18T17:00, 2026-11-18T18:00, 2026-11-18T19:00, 2026-11-18T20:00, 2026-11-18T21:00, 2026-11-18T22:00, 2026-11-18T23:00, 2026-11-19T00:00, 2026-11-19T01:00, 2026-11-19T02:00, 2026-11-19T03:00, 2026-11-19T04:00, 2026-11-19T05:00, 2026-11-19T06:00, 2026-11-19T07:00, 2026-11-19T08:00, 2026-11-19T09:00, 2026-11-19T10:00, 2026-11-19T11:00, 2026-11-19T12:00, 2026-11-19T13:00, 2026-11-19T14:00, 2026-11-19T15:00, 2026-11-19T16:00, 2026-11-19T17:00, 2026-11-19T18:00, 2026-11-19T19:00, 2026-11-19T20:00, 2026-11-19T21:00, 2026-11-19T22:00, 2026-11-19T23:00), List(5.4, 5.9, 9.0, 8.0, 8.0, 8.3, 8.8, 8.3, 7.6, 9.4, 9.7, 9.3, 9.2, 7.7, 7.3, 6.5, 7.9, 8.6, 8.6, 10.8, 11.0, 7.6, 9.0, 9.3, 9.5, 9.5, 8.0, 8.1, 9.3, 9.7, 9.9, 9.4, 9.7, 8.9, 8.2, 9.4, 9.2, 7.5, 5.0, 3.9, 4.8, 7.8, 9.2, 12.0, 13.4, 10.2, 10.0, 9.8))","List(mm, %, °C, iso8601, km/h)",40.710335,-73.99308,"List(New York, 2026-11-18T03:34:22.347594+00:00)",GMT,GMT,0,2026-11-18T03:40:06.506Z,dbfs:/Workspace/Users/cecp08@gmail.com/landing_zone/weather_raw_20261118_033422.json
16.0,0.06222724914550781,"List(List(0.0, 0.4, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.6, 0.0, 0.0, 0.1, 0.3, 0.2, 0.1, 0.0, 1.2, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.4, 2.0, 2.7, 4.3, 2.3, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.4, 0.4, 1.4, 1.4, 1.3), List(81, 88, 87, 85, 84, 83, 85, 84, 86, 92, 88, 82, 80, 84, 83, 84, 78, 88, 89, 89, 86, 88, 85, 87, 90, 96, 96, 94, 93, 92, 92, 91, 91, 89, 87, 85, 82, 78, 75, 75, 77, 79, 80, 89, 90, 92, 94, 95), List(9.7, 9.7, 10.0, 9.8, 9.9, 10.0, 9.8, 9.6, 9.6, 9.5, 9.7, 10.6, 11.1, 10.9, 11.2, 11.2, 11.3, 9.9, 9.7, 9.6, 9.3, 8.7, 8.9, 8.9, 8.9, 8.9, 8.9, 8.5, 8.4, 8.3, 8.3, 8.3, 8.1, 8.2, 8.7, 9.1, 9.6, 10.1, 10.4, 10.4, 10.0, 9.6, 9.8, 9.6, 9.4, 9.2, 9.0, 9.0), List(2026-11-18T00:00, 2026-11-18T01:00, 2026-11-18T02:00, 2026-11-18T03:00, 2026-11-18T04:00, 2026-11-18T05:00, 2026-11-18T06:00, 2026-11-18T07:00, 2026-11-18T08:00, 2026-11-18T09:00, 2026-11-18T10:00, 2026-11-18T11:00, 2026-11-18T12:00, 2026-11-18T13:00, 2026-11-18T14:00, 2026-11-18T15:00, 2026-11-18T16:00, 2026-11-18T17:00, 2026-11-18T18:00, 2026-11-18T19:00, 2026-11-18T20:00, 2026-11-18T21:00, 2026-11-18T22:00, 2026-11-18T23:00, 2026-11-19T00:00, 2026-11-19T01:00, 2026-11-19T02:00, 2026-11-19T03:00, 2026-11-19T04:00, 2026-11-19T05:00, 2026-11-19T06:00, 2026-11-19T07:00, 2026-11-19T08:00, 2026-11-19T09:00, 2026-11-19T10:00, 2026-11-19T11:00, 2026-11-19T12:00, 2026-11-19T13:00, 2026-11-19T14:00, 2026-11-19T15:00, 2026-11-19T16:00, 2026-11-19T17:00, 2026-11-19T18:00, 2026-11-19T19:00, 2026-11-19T20:00, 2026-11-19T21:00, 2026-11-19T22:00, 2026-11-19T23:00), List(14.0, 14.8, 16.6, 16.2, 16.6, 17.3, 14.8, 13.7, 14.4, 12.2, 14.8, 15.5, 16.6, 15.8, 14.0, 7.9, 13.7, 11.2, 4.7, 5.0, 6.8, 2.9, 4.3, 4.0, 7.2, 6.8, 7.6,

Silver Layer

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_bronze = spark.table("workspace.default.bronze_weather_raw")

# 1. Zip parallel hourly arrays together and explode into individual rows
df_exploded = df_bronze.select(
    F.col("metadata.city").alias("city_name"),
    F.col("latitude").cast("decimal(9,6)").alias("latitude"),
    F.col("longitude").cast("decimal(9,6)").alias("longitude"),
    F.col("elevation").cast("decimal(8,2)").alias("elevation"),
    F.col("timezone").alias("timezone"),
    F.col("metadata.ingested_at_utc").cast("timestamp").alias("ingested_at"),
    F.explode(
        F.arrays_zip(
            F.col("hourly.time"),
            F.col("hourly.temperature_2m"),
            F.col("hourly.relative_humidity_2m"),
            F.col("hourly.precipitation"),
            F.col("hourly.wind_speed_10m"),
        )
    ).alias("hourly_record"),
)

# 2. Flatten struct fields and enforce data types
df_cleaned = df_exploded.select(
    "city_name",
    "latitude",
    "longitude",
    "elevation",
    "timezone",
    F.to_timestamp("hourly_record.time").alias("forecast_timestamp"),
    F.col("hourly_record.temperature_2m").cast("decimal(5,2)").alias("temperature_2m"),
    F.col("hourly_record.relative_humidity_2m").cast("int").alias("relative_humidity_2m"),
    F.col("hourly_record.precipitation").cast("decimal(6,2)").alias("precipitation"),
    F.col("hourly_record.wind_speed_10m").cast("decimal(6,2)").alias("wind_speed_10m"),
    "ingested_at",
).filter(F.col("forecast_timestamp").isNotNull())

# 3. Deduplicate within the incoming batch (keep latest ingested_at per location + timestamp)
window_spec = Window.partitionBy("latitude", "longitude", "forecast_timestamp").orderBy(
    F.col("ingested_at").desc()
)
df_deduped = (
    df_cleaned.withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

# 4. Upsert (MERGE) into Silver Delta Table
silver_table_name = "workspace.default.silver_weather_observations"

if not spark.catalog.tableExists(silver_table_name):
    df_deduped.write.format("delta").saveAsTable(silver_table_name)
else:
    silver_delta = DeltaTable.forName(spark, silver_table_name)
    (
        silver_delta.alias("target")
        .merge(
            df_deduped.alias("source"),
            """
            target.latitude = source.latitude AND
            target.longitude = source.longitude AND
            target.forecast_timestamp = source.forecast_timestamp
            """,
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

display(spark.table(silver_table_name))


city_name,latitude,longitude,elevation,timezone,forecast_timestamp,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,ingested_at
Tokyo,35.700000,139.625000,37.00,GMT,2026-11-18T00:00:00.000Z,12.60,83,0.00,1.50,2026-11-18T03:34:23.394Z
Tokyo,35.700000,139.625000,37.00,GMT,2026-11-18T01:00:00.000Z,14.40,77,0.00,1.30,2026-11-18T03:34:23.394Z
Tokyo,35.700000,139.625000,37.00,GMT,2026-11-18T02:00:00.000Z,16.50,70,0.00,1.30,2026-11-18T03:34:23.394Z
Tokyo,35.700000,139.625000,37.00,GMT,2026-11-18T03:00:00.000Z,18.40,67,0.00,3.70,2026-11-18T03:34:23.394Z
Tokyo,35.700000,139.625000,37.00,GMT,2026-11-18T04:00:00.000Z,19.20,66,0.00,4.00,2026-11-18T03:34:23.394Z
Tokyo,35.700000,139.625000,37.00,GMT,2026-11-18T05:00:00.000Z,19.30,69,0.00,4.90,2026-11-18T03:34:23.394Z
Tokyo,35.700000,139.625000,37.00,GMT,2026-11-18T06:00:00.000Z,18.70,74,0.00,4.70,2026-11-18T03:34:23.394Z
Tokyo,35.700000,139.625000,37.00,GMT,2026-11-18T07:00:00.000Z,17.90,78,0.00,3.20,2026-11-18T03:34:23.394Z
Tokyo,35.700000,139.625000,37.00,GMT,2026-11-18T08:00:00.000Z,16.60,85,0.00,3.40,2026-11-18T03:34:23.394Z
Tokyo,35.700000,139.625000,37.00,GMT,2026-11-18T09:00:00.000Z,15.90,91,0.00,3.80,2026-11-18T03:34:23.394Z


Gold Layer

In [0]:
from pyspark.sql import functions as F

df_silver = spark.table("workspace.default.silver_weather_observations")

# 1. Dimension Table: dim_location
df_dim_location = (
    df_silver.select("city_name", "latitude", "longitude", "elevation", "timezone")
    .distinct()
    .withColumn(
        "location_key",
        F.abs(F.hash("latitude", "longitude")),  # Surrogate key
    )
)
df_dim_location.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_dim_location"
)

# 2. Dimension Table: dim_date
df_dim_date = (
    df_silver.select(F.to_date("forecast_timestamp").alias("forecast_date"))
    .distinct()
    .withColumn("date_key", F.date_format("forecast_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("forecast_date"))
    .withColumn("month", F.month("forecast_date"))
    .withColumn("day", F.dayofmonth("forecast_date"))
    .withColumn("day_of_week", F.date_format("forecast_date", "EEEE"))
)
df_dim_date.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_dim_date"
)

# 3. Fact Table: fact_weather_forecast
df_fact = (
    df_silver.withColumn("location_key", F.abs(F.hash("latitude", "longitude")))
    .withColumn("date_key", F.date_format("forecast_timestamp", "yyyyMMdd").cast("int"))
    .select(
        "location_key",
        "date_key",
        "forecast_timestamp",
        F.hour("forecast_timestamp").alias("forecast_hour"),
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "wind_speed_10m",
        "ingested_at",
    )
)
df_fact.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_fact_weather_forecast"
)

SQL

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.locations (
    location_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    city_name VARCHAR(100) NOT NULL,
    latitude DECIMAL(9, 6) NOT NULL,
    longitude DECIMAL(9, 6) NOT NULL,
    elevation DECIMAL(8, 2),
    timezone VARCHAR(50),
    UNIQUE(latitude, longitude)
);

CREATE TABLE IF NOT EXISTS workspace.default.weather_observations (
    observation_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    location_id BIGINT NOT NULL,
    forecast_timestamp TIMESTAMP NOT NULL,
    temperature_2m DECIMAL(5, 2),
    relative_humidity_2m INTEGER,
    precipitation DECIMAL(6, 2),
    wind_speed_10m DECIMAL(6, 2),
    ingested_at TIMESTAMP NOT NULL,
    FOREIGN KEY (location_id) REFERENCES workspace.default.locations(location_id),
    UNIQUE(location_id, forecast_timestamp)
);

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS weather_db;
CREATE VOLUME IF NOT EXISTS weather_db.landing_zone;